In [2]:
# 원본 경로
DRIVE_DIR = "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/그래빗(GrabIT)/PP-PicoDet"
ZIP_PATH  = f"{DRIVE_DIR}/PP-PicoDet.zip"

# 로컬 대상 경로
LOCAL_ZIP  = "/content/YOLOX-nano.zip"

# ZIP 복사 (진행률 표시)
!rsync -ah --info=progress2 "{ZIP_PATH}" "{LOCAL_ZIP}"

        628.51M 100%  691.30MB/s    0:00:00 (xfr#1, to-chk=0/1)


In [3]:
import os
import zipfile

In [4]:
LOCAL_ZIP = "/content/YOLOX-nano.zip"
EXTRACT_DIR = "/content/YOLOX-nano_dataset"
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(LOCAL_ZIP, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("✅ 압축 해제 완료:", EXTRACT_DIR)


✅ 압축 해제 완료: /content/YOLOX-nano_dataset


In [5]:
!ls -R {EXTRACT_DIR} | head -n 20

/content/YOLOX-nano_dataset:
annotations
custom_picodet.yml
images_640

/content/YOLOX-nano_dataset/annotations:
instances_test.json
instances_train.json
instances_val.json

/content/YOLOX-nano_dataset/images_640:
test
train
val

/content/YOLOX-nano_dataset/images_640/test:
test_00000.jpg
test_00001.jpg
test_00002.jpg
test_00003.jpg


In [6]:
# 1. YOLOX 복제 및 설치
!git clone https://github.com/Megvii-BaseDetection/YOLOX.git
%cd YOLOX
!pip install -r requirements.txt
!pip install -v -e . --no-build-isolation
!pip install onnx onnx-simplifier onnxruntime onnx2tf

# 2. PyTorch 2.6+ 가중치 로딩 버그 패치
import pathlib, re
p = pathlib.Path("yolox/core/trainer.py")
txt = p.read_text()
txt2 = re.sub(r"torch\.load\((ckpt_file,\s*map_location=self\.device)\)", r"torch.load(\1, weights_only=False)", txt)
p.write_text(txt2)

# 3. 사전 학습 가중치 다운로드
!wget https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth

Cloning into 'YOLOX'...
remote: Enumerating objects: 1940, done.
remote: Total 1940 (delta 0), reused 0 (delta 0), pack-reused 1940 (from 1)
Receiving objects: 100% (1940/1940), 7.55 MiB | 32.47 MiB/s, done.
Resolving deltas: 100% (1166/1166), done.
/content/YOLOX
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 38.7 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Obtaining file:///content/YOLOX
  Running command python setup.py egg_info
  /usr/local/l

In [7]:
# COCO 폴더 생성
!mkdir -p datasets/COCO/annotations

# 팀장님의 압축 해제 경로를 COCO 구조로 링크 (상세 경로는 ls 결과에 따라 맞춰야 합니다)
# 아래는 일반적인 YOLOX 데이터 구조 기준입니다.
!ln -sf /content/YOLOX-nano_dataset/images_640/train datasets/COCO/train2017
!ln -sf /content/YOLOX-nano_dataset/images_640/val   datasets/COCO/val2017
!ln -sf /content/YOLOX-nano_dataset/train.json datasets/COCO/annotations/instances_train2017.json
!ln -sf /content/YOLOX-nano_dataset/val.json   datasets/COCO/annotations/instances_val2017.json

In [8]:
!ls -l datasets/COCO/train2017

lrwxrwxrwx 1 root root 44 Feb  3 05:41 datasets/COCO/train2017 -> /content/YOLOX-nano_dataset/images_640/train


In [9]:
drive_save_path = '/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/그래빗(GrabIT)/YOLOX/YOLOX-nano'

In [10]:
!rm -rf YOLOX_outputs
os.symlink(drive_save_path, 'YOLOX_outputs')

In [36]:
!pip install loguru -q
!pip install thop -q
!pip install ninja -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 5.4 MB/s eta 0:00:00


In [13]:
import os
from yolox.exp import Exp as MyExp

In [16]:
%cd /content/YOLOX

/content/YOLOX


In [14]:
class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()
        # --- 기본 설정 유지 ---
        self.num_classes = 53
        self.depth = 0.33
        self.width = 0.25
        self.exp_name = "yolox_nano_standard"

        # --- 팀장님 요청: 100에폭, 640 해상도 ---
        self.max_epoch = 100
        self.input_size = (640, 640)

        # --- [중요] 정확도 측정 주기 설정 ---
        # 1로 설정하면 매 에폭마다 mAP50, mAP50-95를 계산하여 출력합니다.
        self.eval_interval = 5

        # --- 데이터 경로 (이전 답변의 심볼릭 링크 기준) ---
        self.data_dir = "datasets/COCO"
        self.train_ann = "instances_train2017.json"
        self.val_ann = "instances_val2017.json"

        # 기본 증강 설정 유지
        self.mosaic_prob = 1.0
        self.mixup_prob = 1.0
        self.no_aug_epochs = 15
        self.print_interval = 10

In [19]:
# 설계도 폴더 생성
os.makedirs("exps/custom", exist_ok=True)

# 53개 클래스, 640 해상도, 100에폭, 5주기 mAP 출력 설정
exp_content = """
import os
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()
        self.num_classes = 53
        self.depth = 0.33
        self.width = 0.25
        self.exp_name = "yolox_nano_standard"

        # 학습 설정 (팀장님 요청 사항)
        self.max_epoch = 100
        self.input_size = (640, 640)
        self.eval_interval = 5

        # 데이터 경로
        self.data_dir = "datasets/COCO"
        self.train_ann = "instances_train2017.json"
        self.val_ann = "instances_val2017.json"

        # 기본 증강 및 기타 설정 (SiLU 유지)
        self.mosaic_prob = 1.0
        self.mixup_prob = 1.0
        self.no_aug_epochs = 15
        self.print_interval = 10
"""

with open("exps/custom/yolox_nano_standard.py", "w") as f:
    f.write(exp_content.strip())

print("✅ exps/custom/yolox_nano_standard.py 생성 완료")

✅ exps/custom/yolox_nano_standard.py 생성 완료


In [27]:
# 1. 기존 잘못된 링크 싹 정리
!rm -rf datasets/COCO/annotations/*

In [26]:
search_root = "/content/YOLOX-nano_dataset"
for root, dirs, files in os.walk(search_root):
    for file in files:
        if file.endswith(".json"):
            print(f"📍 발견된 경로: {os.path.join(root, file)}")

📍 발견된 경로: /content/YOLOX-nano_dataset/annotations/instances_test.json
📍 발견된 경로: /content/YOLOX-nano_dataset/annotations/instances_train.json
📍 발견된 경로: /content/YOLOX-nano_dataset/annotations/instances_val.json


In [28]:
!ln -sf /content/YOLOX-nano_dataset/annotations/instances_train.json datasets/COCO/annotations/instances_train2017.json
!ln -sf /content/YOLOX-nano_dataset/annotations/instances_val.json  datasets/COCO/annotations/instances_val2017.json

# 3. 이미지 폴더도 똑같이 확인 후 연결
!ln -sf /content/YOLOX-nano_dataset/images_640/train datasets/COCO/train2017
!ln -sf /content/YOLOX-nano_dataset/images_640/val   datasets/COCO/val2017

In [29]:
!ls -l datasets/COCO/annotations/

total 4
lrwxrwxrwx 1 root root 60 Feb  3 06:01 instances_train2017.json -> /content/YOLOX-nano_dataset/annotations/instances_train.json
lrwxrwxrwx 1 root root 58 Feb  3 06:01 instances_val2017.json -> /content/YOLOX-nano_dataset/annotations/instances_val.json


In [31]:
!head -n 20 datasets/COCO/annotations/instances_train2017.json

{
    "images": [
        {
            "id": 0,
            "file_name": "train_00000.jpg",
            "width": 640,
            "height": 640
        },
        {
            "id": 1,
            "file_name": "train_00001.jpg",
            "width": 640,
            "height": 640
        },
        {
            "id": 2,
            "file_name": "train_00002.jpg",
            "width": 640,
            "height": 640
        },


In [38]:
!PYTHONPATH=. python tools/train.py \
    -f exps/custom/yolox_nano_standard.py \
    -d 1 \
    -b 128 \
    --fp16 \
    --cache \
    -c '/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/그래빗(GrabIT)/YOLOX/YOLOX-nano/yolox_nano_standard/latest_ckpt.pth'

2026-02-03 06:25:07.229542: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-03 06:25:07.248326: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770099907.271056   16434 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770099907.277776   16434 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770099907.295145   16434 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [39]:
import random
import shutil

In [55]:
# 팀장님의 53개 클래스 리스트 (결과 해석.txt 기반)
custom_class_names = (
    "Haitai Pocky Blueberry 41g", "Lotte Kkokkalcorn Savory Taste 72g", "Nongshim Squid House 83g",
    "Nongshim Spicy Shrimp Crackers 90g", "Crown Corn Cho 66g", "Nongshim Banana Kick 75g",
    "Lotte Xylitol Beta Vita D 86g", "Ricola Swiss Candy Cranberry 40g", "Cavendish & Harvey Tropical Fruit Drops 200g",
    "Lotte Dream Cacao 82 (GABA) 86g", "Nongshim Pringles Classic 53g", "Lotte Fluffy Malang Cow Strawberry Milk 158g",
    "Lotte ABC Milk Chocolate 65g", "Lotte Stone Age 90g", "Lotte Squid Peanut 90g", "Nongshim Chip Potato Original 125g",
    "Binggrae Smoky Bacon Chip Spicy 70g", "Haitai Matdongsan 90g", "Orion Poka Chip Original 66g", "Cheongwoo Lemon Candy 100g",
    "Popo Candy Ice Flavor", "Big Wadadak Cola Flavor", "Nemo Snack Spicy Flavor", "Tipo Green Tea Cookie",
    "Crown Mychew Strawberry 92g", "Woongjin Morning Rice 500ml", "Ilhwa McCol 500ml", "Lotte Sarangcho Toktok Sparkling 330ml",
    "Georgia Gotica Vintage Black 390ml", "Haitai Galbae Cider 238ml", "Lotte Lets Be 175ml", "Coca-Cola Kin Cider 185ml",
    "Coca-Cola Fanta Grape 250ml", "Lotte Milkis 500ml", "Lotte Corn Silk Tea 500ml", "Coca-Cola Sprite 500ml",
    "Lotte Farmers Juice Bar Orange 1L", "Kwangdong Fermented Red Ginseng 100ml", "Kwangdong Red Ginseng Ssanghwajin 100ml",
    "Lotte Hot6 355ml", "Woongjin Nature Jeju Tangerine 340ml", "Lotte Cantata Sweet Americano 275ml",
    "Monster Energy Pipeline Punch 355ml", "Seoul Milk Morning Juice Grape 950ml", "Namyang Choco Emong 180ml",
    "Lotte Only Price Mineral Water", "Lotte Del Monte Cold Apple Juice", "Lotte Pepsi 160ml", "Nongshim Baeksansu 330ml",
    "Dongsuh Maxim TOP Master Latte", "Cheetos BBQ Flavor", "Cheetos Spicy Sweet Flavor", "Cheetos Real Corn Soup Flavor"
)

# YOLOX 내부의 클래스 정의 파일을 강제로 수정합니다.
class_file_path = "/content/YOLOX/yolox/data/datasets/coco_classes.py"

with open(class_file_path, "w") as f:
    f.write(f"COCO_CLASSES = {str(custom_class_names)}")

print("✅ 클래스 이름표(Label Map)가 53개 커스텀 리스트로 업데이트되었습니다!")

✅ 클래스 이름표(Label Map)가 53개 커스텀 리스트로 업데이트되었습니다!


In [56]:
# PyTorch 보안 해제 및 GUI 코드 무력화
!sed -i 's/torch.load(ckpt_file, map_location="cpu")/torch.load(ckpt_file, map_location="cpu", weights_only=False)/g' /content/YOLOX/tools/demo.py
!sed -i 's/cv2.imshow/# cv2.imshow/g' /content/YOLOX/tools/demo.py
!sed -i 's/ch = cv2.waitKey(0)/ch = 0/g' /content/YOLOX/tools/demo.py
!sed -i 's/if ch == 27/if False:/g' /content/YOLOX/tools/demo.py

In [57]:
import os
import random
import glob

In [60]:
# 설정
test_dir = "/content/YOLOX-nano_dataset/images_640/test"
best_model = "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/그래빗(GrabIT)/YOLOX/YOLOX-nano/yolox_nano_standard/best_ckpt.pth"
exp_file = "exps/custom/yolox_nano_standard.py"

# 랜덤 10장 선정
all_images = [os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.lower().endswith(('.jpg', '.png'))]
selected_images = random.sample(all_images, 10)

print(f"🚀 총 10장의 이미지 분석을 시작합니다...")

for i, img_path in enumerate(selected_images):
    print(f"[{i+1}/10] 분석 중: {os.path.basename(img_path)}")
    # 한 장씩 독립적으로 실행하여 충돌 방지
    !PYTHONPATH=. python tools/demo.py image \
        -f {exp_file} \
        -c '{best_model}' \
        --path '{img_path}' \
        --conf 0.4 \
        --nms 0.45 \
        --tsize 640 \
        --save_result \
        --device gpu > /dev/null 2>&1  # 로그 간소화

print("\n✅ 10장 모두 분석 완료!")

🚀 총 10장의 이미지 분석을 시작합니다...
[1/10] 분석 중: test_00031.jpg
[2/10] 분석 중: test_00346.jpg
[3/10] 분석 중: test_00040.jpg
[4/10] 분석 중: test_00009.jpg
[5/10] 분석 중: test_00108.jpg
[6/10] 분석 중: test_00372.jpg
[7/10] 분석 중: test_00290.jpg
[8/10] 분석 중: test_00249.jpg
[9/10] 분석 중: test_00227.jpg
[10/10] 분석 중: test_00205.jpg

✅ 10장 모두 분석 완료!


In [42]:
import matplotlib.pyplot as plt
from PIL import Image

In [61]:
result_files = sorted(glob.glob("./YOLOX_outputs/yolox_nano_standard/vis_res/*/*.jpg"), key=os.path.getmtime)
latest_10 = result_files[-10:] # 가장 최근 10장

plt.figure(figsize=(25, 12))
for i, img_path in enumerate(latest_10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(Image.open(img_path))
    plt.axis('off')
    plt.title(os.path.basename(img_path), fontsize=10)

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.